# Cross-Dataset Analysis: Calibra Observability Metrics

**Experiment**: Profile 15 public robot imitation-learning datasets across four
deterministic observability metrics and test whether the resulting empirical
distributions confirm, challenge, or falsify the Calibra claim registry.

**Datasets profiled (15 total)**
- ALOHA family: 10 datasets (sim/hardware × human/scripted × static/mobile)
- DROID-100: diverse 7-DOF hardware teleoperation, 15 Hz
- BridgeData V2: 50 k-episode velocity-command dataset, 5 Hz
- PushT / PushT-image: 2-D velocity-command simulation, ~10 Hz
- SVLA SO-100 pick-place & stacking: 5-DOF low-cost arm, ~50 Hz

**Metrics**
| Metric | Symbol | Interpretation |
|--------|--------|----------------|
| Jerk spike rate | `spike_rate` | Fraction of steps where jerk norm > 5× episode median |
| Velocity discontinuity rate | `vel_disc_rate` | Fraction of steps with >20% peak velocity change |
| Timing jitter CV | `jitter_cv` | Coefficient of variation of inter-step intervals |
| Log dimensionless jerk | `ldlj` | Smoothness index; more-negative = less smooth |

All numbers come from pre-computed reference profiles in `calibra/references/`.
No training or external API calls are required to reproduce this notebook.


In [ ]:
import json, glob, pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

REPO_ROOT = pathlib.Path("..")
REF_DIR   = REPO_ROOT / "calibra" / "references"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
COLORS = {"position": "#2196F3", "velocity": "#FF5722", "scripted": "#9C27B0"}

def get_metric(dist, *keys):
    prefixes = [
        "control_smoothness/", "jerk_spikes/",
        "velocity_discontinuity/", "temporal_stability/", "",
    ]
    for key in keys:
        for pfx in prefixes:
            v = dist.get(pfx + key, {})
            if v and v.get("mean") is not None:
                return v
    return {}

rows = []
for f in sorted(REF_DIR.glob("*.json")):
    d = json.loads(f.read_text())
    meta = d.get("meta", {})
    dist = d.get("per_episode_distributions", {})

    def m(key):
        return get_metric(dist, key)

    rows.append(dict(
        dataset    = meta.get("dataset", f.stem),
        short      = f.stem.replace("_", " "),
        n_episodes = meta.get("n_episodes"),
        ctrl       = meta.get("control_mode", "?"),
        is_scripted= "scripted" in f.stem,
        spike_mean = m("per_episode_spike_rate").get("mean"),
        spike_p95  = m("per_episode_spike_rate").get("p95"),
        vel_mean   = m("per_episode_vel_disc_rate").get("mean"),
        vel_p95    = m("per_episode_vel_disc_rate").get("p95"),
        jitter_mean= m("per_episode_jitter_cv").get("mean"),
        ldlj_mean  = m("per_episode_ldlj").get("mean"),
    ))

df = pd.DataFrame(rows)
print(f"{len(df)} reference profiles loaded")
df[["short", "ctrl", "n_episodes", "spike_mean", "vel_mean"]].round(4)

## Figure 1 — Jerk Spike Rate Across All Datasets

**Hypothesis (JS-001):** Human-collected position-command datasets have spike rate < 2%;
scripted datasets regularly exceed this.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

df_spike = df.dropna(subset=["spike_mean"]).sort_values("spike_mean")

colors = [
    COLORS["scripted"] if r.is_scripted else COLORS[r.ctrl]
    for _, r in df_spike.iterrows()
]
bars = ax.barh(
    df_spike["short"], df_spike["spike_mean"] * 100,
    color=colors, alpha=0.85, edgecolor="white", linewidth=0.5,
)

# P95 whiskers
ax.hlines(
    y=range(len(df_spike)),
    xmin=df_spike["spike_mean"] * 100,
    xmax=df_spike["spike_p95"].fillna(df_spike["spike_mean"]) * 100,
    colors="gray", linewidth=1.5, alpha=0.6,
)

ax.axvline(2, color="crimson", linestyle="--", linewidth=1.2, label="JS-001 threshold (2%)")
ax.axvline(4, color="darkorange", linestyle=":",  linewidth=1.2, label="JS-001 falsification (4%)")
ax.set_xlabel("Mean jerk spike rate (%)")
ax.set_title("Jerk Spike Rate: 15 public datasets", fontweight="bold")

legend_handles = [
    mpatches.Patch(color=COLORS["position"], label="position-command"),
    mpatches.Patch(color=COLORS["velocity"],  label="velocity-command"),
    mpatches.Patch(color=COLORS["scripted"],  label="scripted/planner"),
    plt.Line2D([0],[0], color="crimson",    linestyle="--", label="JS-001 threshold (2%)"),
    plt.Line2D([0],[0], color="darkorange", linestyle=":",  label="falsification boundary (4%)"),
]
ax.legend(handles=legend_handles, fontsize=8, loc="lower right")
ax.annotate("Gray bars = P95", xy=(0.98, 0.02), xycoords="axes fraction",
            ha="right", fontsize=7, color="gray")

fig.tight_layout()
fig.savefig("figures/fig1_jerk_spike_rate.pdf", bbox_inches="tight")
fig.savefig("figures/fig1_jerk_spike_rate.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved fig1")

## Figure 2 — Velocity Discontinuity Rate: Control Frequency Effect

**Hypothesis (VD-001/VD-003):** Position-command datasets < 5%; velocity-command
at ~10 Hz in 10–20%; velocity-command at ≤5 Hz exceeds 50%.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

df_vel = df.dropna(subset=["vel_mean"]).sort_values("vel_mean")

colors = [
    COLORS["scripted"] if r.is_scripted else COLORS[r.ctrl]
    for _, r in df_vel.iterrows()
]
ax.barh(
    df_vel["short"], df_vel["vel_mean"] * 100,
    color=colors, alpha=0.85, edgecolor="white", linewidth=0.5,
)
ax.hlines(
    y=range(len(df_vel)),
    xmin=df_vel["vel_mean"] * 100,
    xmax=df_vel["vel_p95"].fillna(df_vel["vel_mean"]) * 100,
    colors="gray", linewidth=1.5, alpha=0.6,
)

ax.axvline(5,  color="steelblue",  linestyle="--", linewidth=1.2, label="VD-001 threshold (5%): position")
ax.axvline(20, color="#FF5722",    linestyle="--", linewidth=1.2, label="VD-003 upper bound (20%): ~10 Hz vel")
ax.axvline(50, color="darkred",    linestyle=":",  linewidth=1.2, label="VD-003 lower bound (50%): ≤5 Hz vel")

ax.set_xlabel("Mean velocity discontinuity rate (%)")
ax.set_title("Velocity Discontinuity Rate: control-frequency stratification", fontweight="bold")

handles = [
    mpatches.Patch(color=COLORS["position"], label="position-command"),
    mpatches.Patch(color=COLORS["velocity"],  label="velocity-command"),
    mpatches.Patch(color=COLORS["scripted"],  label="scripted/planner"),
    plt.Line2D([0],[0], color="steelblue",  linestyle="--", label="VD-001 threshold (5%)"),
    plt.Line2D([0],[0], color="#FF5722",    linestyle="--", label="VD-003 upper ~10 Hz (20%)"),
    plt.Line2D([0],[0], color="darkred",    linestyle=":",  label="VD-003 lower ≤5 Hz (50%)"),
]
ax.legend(handles=handles, fontsize=8, loc="lower right")

fig.tight_layout()
fig.savefig("figures/fig2_velocity_discontinuity.pdf", bbox_inches="tight")
fig.savefig("figures/fig2_velocity_discontinuity.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved fig2")

## Figure 3 — Smoothness vs Quality: LDLJ Scatter

LDLJ (Log Dimensionless Jerk) is a scale-invariant smoothness metric.
More-negative = less smooth (rougher motion).
We plot LDLJ against spike rate to show they are correlated but not redundant.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

df_s = df.dropna(subset=["spike_mean", "ldlj_mean"])

colors = [
    COLORS["scripted"] if r.is_scripted else COLORS[r.ctrl]
    for _, r in df_s.iterrows()
]
sc = ax.scatter(
    df_s["ldlj_mean"], df_s["spike_mean"] * 100,
    c=colors, s=df_s["n_episodes"].clip(upper=500) / 3,
    alpha=0.8, edgecolors="white", linewidths=0.5,
)

for _, r in df_s.iterrows():
    lbl = r["short"][:20]
    ax.annotate(lbl, (r["ldlj_mean"], r["spike_mean"] * 100),
                xytext=(4, 2), textcoords="offset points", fontsize=6.5, color="#333")

ax.set_xlabel("Mean LDLJ (more negative = less smooth)")
ax.set_ylabel("Mean jerk spike rate (%)")
ax.set_title("Smoothness (LDLJ) vs Jerk Spike Rate", fontweight="bold")
ax.annotate("Bubble size ∝ min(n_episodes, 500)", xy=(0.02, 0.96),
            xycoords="axes fraction", fontsize=7, color="gray", va="top")

handles = [
    mpatches.Patch(color=COLORS["position"], label="position-command"),
    mpatches.Patch(color=COLORS["velocity"],  label="velocity-command"),
    mpatches.Patch(color=COLORS["scripted"],  label="scripted/planner"),
]
ax.legend(handles=handles, fontsize=8)

fig.tight_layout()
fig.savefig("figures/fig3_ldlj_vs_spike.pdf", bbox_inches="tight")
fig.savefig("figures/fig3_ldlj_vs_spike.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved fig3")

## Figure 4 — Claim Status Summary

How many claims are validated, active, or falsified across each metric family?

In [ ]:
CLAIMS_DIR = REPO_ROOT / "calibra" / "claims"
STATUS_COLORS = {
    "validated": "#4CAF50",
    "active_hypothesis": "#2196F3",
    "provisional": "#FF9800",
    "falsified": "#F44336",
    "updated": "#9C27B0",
}

all_claims = []
for f in sorted(CLAIMS_DIR.glob("*.json")):
    d = json.loads(f.read_text())
    for c in d.get("claims", []):
        all_claims.append(dict(
            id      = c["id"],
            metric  = c["metric"],
            status  = c["status"],
            n_ev    = len(c.get("evidence", [])),
            conf    = c.get("confidence", "?"),
        ))

cdf = pd.DataFrame(all_claims)
print(f"{len(cdf)} claims loaded")

pivot = cdf.groupby(["metric", "status"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4))
pivot.plot(kind="bar", ax=ax, color=[STATUS_COLORS.get(c, "gray") for c in pivot.columns],
           edgecolor="white", linewidth=0.5)
ax.set_xlabel("Metric family")
ax.set_ylabel("Number of claims")
ax.set_title("Claim Registry Status by Metric", fontweight="bold")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Status", fontsize=8, title_fontsize=8)

fig.tight_layout()
fig.savefig("figures/fig4_claim_status.pdf", bbox_inches="tight")
fig.savefig("figures/fig4_claim_status.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved fig4")

## Summary Table (LaTeX-ready)

In [ ]:
print("\n=== Summary Table ===")
summary = df[[
    "short", "ctrl", "n_episodes",
    "spike_mean", "vel_mean", "jitter_mean", "ldlj_mean"
]].copy()
summary["spike_%"]    = (summary.pop("spike_mean") * 100).round(2)
summary["vel_disc_%"] = (summary.pop("vel_mean")   * 100).round(2)
summary["jitter_cv"]  = summary.pop("jitter_mean").map(lambda x: f"{x:.1e}" if x else "-")
summary["ldlj"]       = summary.pop("ldlj_mean").round(2)
print(summary.to_string(index=False))

# LaTeX table
latex = summary.to_latex(index=False, float_format="%.2f",
    caption="Calibra observability metrics across 15 public imitation-learning datasets.",
    label="tab:cross_dataset")

out_path = pathlib.Path("figures/table1_cross_dataset.tex")
out_path.write_text(latex)
print(f"\nLaTeX table written to {out_path}")

## Key Findings

1. **Scripted planners 25× noisier than human teleop** — ALOHA scripted datasets
   show 22–25% jerk spike rate vs <1% for human-collected counterparts. Policy
   training on scripted data may be inadvertently learning to reproduce
   planner artifacts.

2. **Hardware diversity amplifies noise 10×** — DROID (100 diverse labs) reaches
   4.55% spike rate and 7.1% velocity discontinuity vs ALOHA hardware's <0.5%
   and <1%, respectively. Heterogeneous hardware fleets require per-robot
   calibration before pooling.

3. **Control frequency is a latent confounder** — BridgeData V2 at 5 Hz shows
   80% velocity discontinuity rate — not because the robot is noisy, but because
   coarse time resolution makes smooth paths appear step-like. Claim VD-002
   (10–20% vel_disc for velocity-command datasets) was formally falsified by
   this finding, leading to the frequency-qualified successor VD-003.

4. **Timing quality is uniformly excellent** — jitter CV is below 1e-4 across
   all 15 datasets, suggesting that clock synchronisation in LeRobot v2 is
   reliable. Timestamp dropout is zero in every profile.

5. **LDLJ adds orthogonal signal** — plotting LDLJ against spike rate (Fig. 3)
   shows that the two metrics are correlated for scripted datasets but diverge
   for real hardware, where high spike rate can coexist with moderate LDLJ
   (e.g., DROID). Both metrics are needed for a complete quality picture.
